# 01d — TheSession Irish Folk: Exploratory Data Analysis

This notebook characterises the Irish folk subset (200 tunes converted from ABC notation).
Irish traditional music is primarily monophonic or heterophonic (melody instruments playing
in unison with ornamentation), which should produce distinctly different note density and
pitch range distributions compared to the polyphonic Western Classical subset.

Key EDA dimensions:
- Tune type distribution (reel, jig, hornpipe, etc.) and their rhythmic signatures
- Key/mode distribution (major, dorian, mixolydian — common modes in Irish trad)
- Duration and note density (short tunes — typically 16–32 bars, repeated)
- Pitch class entropy (expected lower than Western Classical due to modal focus)


In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

_here = Path().resolve()
PROJECT_ROOT = _here
for _ in range(5):
    if (PROJECT_ROOT / "PROGRESS.md").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.midi_utils import (
    load_midi, analyse_midi, get_pitch_class_histogram,
    pitch_class_entropy, piano_roll_plot
)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

PC_LABELS = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mohammadashraf/Desktop/Thesis-Best


In [2]:
def build_stats_df(midi_dir, meta_df=None, id_col=None):
    """Analyse all MIDI files in midi_dir; optionally merge metadata."""
    midi_files = sorted(midi_dir.glob("*.mid")) + sorted(midi_dir.glob("*.midi"))
    print(f"Analysing {len(midi_files)} MIDI files ...")
    records = [analyse_midi(p) for p in midi_files]
    df = pd.DataFrame(records)
    if "error" in df.columns:
        bad = df["error"].notna().sum()
        if bad:
            print(f"  Warning: {bad} files failed to load.")
        df = df[df["error"].isna()].drop(columns=["error"])
    df["filename"] = [Path(p).name for p in df["path"]]
    if meta_df is not None and id_col is not None:
        df = df.merge(meta_df, left_on="filename", right_on=id_col, how="left")
    return df

In [3]:
def summary_panel(df, tradition_name, save_path):
    """4-panel summary figure: duration, note density, pitch range, PC entropy."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f"{tradition_name} — EDA Summary (n={len(df)})", fontsize=13, y=1.01)

    # Duration
    ax = axes[0, 0]
    df["duration_s"].div(60).plot.hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
    ax.axvline(df["duration_s"].mean()/60, color="red", linestyle="--", label=f"mean={df['duration_s'].mean()/60:.1f} min")
    ax.set_xlabel("Duration (minutes)")
    ax.set_title("Duration Distribution")
    ax.legend(fontsize=8)

    # Note density
    ax = axes[0, 1]
    df["note_density"].plot.hist(bins=25, ax=ax, color="seagreen", edgecolor="white")
    ax.axvline(df["note_density"].mean(), color="red", linestyle="--", label=f"mean={df['note_density'].mean():.2f}")
    ax.set_xlabel("Notes per second")
    ax.set_title("Note Density Distribution")
    ax.legend(fontsize=8)

    # Pitch range
    ax = axes[1, 0]
    df["pitch_range"].plot.hist(bins=25, ax=ax, color="darkorange", edgecolor="white")
    ax.axvline(df["pitch_range"].mean(), color="red", linestyle="--", label=f"mean={df['pitch_range'].mean():.1f}")
    ax.set_xlabel("Pitch range (semitones)")
    ax.set_title("Pitch Range Distribution")
    ax.legend(fontsize=8)

    # PC entropy
    ax = axes[1, 1]
    df["pc_entropy"].plot.hist(bins=25, ax=ax, color="mediumpurple", edgecolor="white")
    ax.axvline(df["pc_entropy"].mean(), color="red", linestyle="--", label=f"mean={df['pc_entropy'].mean():.3f}")
    ax.set_xlabel("Pitch Class Entropy (bits)")
    ax.set_title("PC Entropy Distribution\n(Yang & Lerch, 2020)")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")


def mean_pc_histogram_plot(df_midi_paths, tradition_name, save_path):
    """Plot the mean pitch class histogram across all pieces."""
    hists = []
    for p in df_midi_paths:
        pm = load_midi(p)
        if pm:
            hists.append(get_pitch_class_histogram(pm))
    if not hists:
        return
    mean_hist = np.mean(hists, axis=0)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(PC_LABELS, mean_hist, color="steelblue", edgecolor="white")
    ax.set_xlabel("Pitch class")
    ax.set_ylabel("Mean relative frequency")
    ax.set_title(f"{tradition_name} — Mean Pitch Class Histogram (n={len(hists)})")
    plt.tight_layout()
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path.relative_to(PROJECT_ROOT)}")

In [4]:
MIDI_DIR = PROJECT_ROOT / "data" / "processed" / "irish_folk" / "midi"
META_CSV = PROJECT_ROOT / "data" / "metadata" / "irish_folk_tunes.csv"

midi_files = list(MIDI_DIR.glob("*.mid"))
if not midi_files:
    raise FileNotFoundError(
        "No MIDI files found. Run notebook 00d_irish_folk_prep.ipynb first."
    )
print(f"Found {len(midi_files)} MIDI files.")

meta = pd.read_csv(META_CSV)
print(f"Metadata rows: {len(meta)}")
meta[["name", "type", "meter", "mode", "tunebooks"]].head(5)

Found 196 MIDI files.
Metadata rows: 200


,name,type,meter,mode,tunebooks
0,Drowsy Maggie,reel,4/4,Edorian,7811
1,Cooley's,reel,4/4,Edorian,6657
2,"Silver Spear, The",reel,4/4,Dmajor,4910
3,"Maid Behind The Bar, The",reel,4/4,Dmajor,4680
4,"Banshee, The",reel,4/4,Gmajor,4461


## 1. Tune type and mode distributions

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_counts = meta["type"].value_counts()
type_counts.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title(f"Tune Type Distribution (n={len(meta)})")
axes[0].set_xlabel("Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

mode_counts = meta["mode"].value_counts().head(15)
mode_counts.plot(kind="bar", ax=axes[1], color="darkorange", edgecolor="white")
axes[1].set_title("Top 15 Key/Mode Distribution")
axes[1].set_xlabel("Mode")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45, labelsize=8)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "eda_irish_type_mode.png"), dpi=150)
plt.show()

## 2. Compute statistics

In [6]:
stats = build_stats_df(MIDI_DIR)
print(f"Files analysed: {len(stats)}")
print("\nDescriptive statistics:")
print(stats[["duration_s", "note_count", "note_density", "pitch_range", "pc_entropy"]].describe().round(3))

Analysing 196 MIDI files ...


Files analysed: 196

Descriptive statistics:
       duration_s  note_count  note_density  pitch_range  pc_entropy
count     196.000     196.000       196.000      196.000     196.000
mean       63.085     225.015         3.609       18.735       2.542
std        28.085     102.706         0.670        4.281       0.155
min        16.000      67.000         1.684       12.000       2.080
25%        48.000     159.750         3.417       17.000       2.442
50%        64.000     222.500         3.625       19.000       2.572
75%        72.000     262.250         3.784       21.000       2.649
max       216.000     768.000         6.604       38.000       2.869


## 3. Distribution plots

In [7]:
summary_panel(stats, "Irish Folk (TheSession)", RESULTS_DIR / "eda_irish_summary.png")

Saved → results/eda_irish_summary.png


## 4. Mean pitch class histogram

In [8]:
mean_pc_histogram_plot(stats["path"].tolist(), "Irish Folk (TheSession)",
                       RESULTS_DIR / "eda_irish_pc_histogram.png")

Saved → results/eda_irish_pc_histogram.png


## 5. PC entropy by tune type

We expect reels (4/4) and hornpipes to have similar pitch entropy, while
slipjigs (9/8) and slides (12/8) may differ due to their distinctive melodic patterns.


In [9]:
# Map filename back to tune metadata for grouping
stats["tune_id"] = stats["filename"].str.extract(r"irish_\d+_(\d+)_").astype(float)
stats_meta = stats.merge(meta[["tune_id", "type", "mode"]], on="tune_id", how="left")

if stats_meta["type"].notna().any():
    entropy_by_type = stats_meta.groupby("type")["pc_entropy"].agg(["mean","std","count"])
    print("PC Entropy by tune type:")
    print(entropy_by_type.sort_values("mean", ascending=False).round(3).to_string())

PC Entropy by tune type:
             mean    std  count
type                           
waltz       2.652  0.115     15
hornpipe    2.618  0.150     14
slip jig    2.597  0.155      7
mazurka     2.583  0.011      2
march       2.566  0.133      6
reel        2.550  0.151     69
jig         2.522  0.140     52
polka       2.519  0.129     15
slide       2.472  0.196      4
strathspey  2.417  0.200      4
barndance   2.401  0.134      6
three-two   2.124  0.062      2


## 6. Sample piano roll

In [10]:
sample_path = stats["path"].iloc[0]
pm = load_midi(sample_path)
sample_name = Path(sample_path).stem[:60]

fig, ax = plt.subplots(figsize=(14, 4))
piano_roll_plot(pm, ax, time_start=0, time_end=30,
                title=f"Piano roll — {sample_name}")
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "eda_irish_piano_roll.png"), dpi=150)
plt.show()

## 7. Save summary statistics

In [11]:
stats["tradition"] = "irish_folk"
stats.to_csv(RESULTS_DIR / "eda_irish_stats.csv", index=False)
print("Saved → results/eda_irish_stats.csv")
print(f"\nPC entropy mean : {stats['pc_entropy'].mean():.3f} bits")
print(f"Note density mean: {stats['note_density'].mean():.2f} notes/s")
print(f"Duration mean    : {stats['duration_s'].mean():.1f} s  ({stats['duration_s'].mean()/60:.2f} min)")

Saved → results/eda_irish_stats.csv

PC entropy mean : 2.542 bits
Note density mean: 3.61 notes/s
Duration mean    : 63.1 s  (1.05 min)
